# Vanilla dense RAG

**Goal:** evaluate a classical RAG pipeline.

System:

```text
question -> dense FAISS retrieval -> top-k context -> generator -> answer
```

We compute:

- answer Exact Match / F1;
- conditional answer perplexity with retrieved context;
- retrieval Hit@k and SupportRecall@k;
- retrieval, generation, and total latency.


In [1]:
%pip install -r ../requirements.txt


[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: pip3.12 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: /Users/polinakorobeinikova/IU/NLP/case-study/case-study-rag-factual-consistency


In [3]:
import json
import time

import pandas as pd
from tqdm.auto import tqdm

from src.config import (
    PROCESSED_DIR, INDEX_DIR, PREDICTIONS_DIR, METRICS_DIR,
    GENERATOR_MODEL, EMBEDDING_MODEL
)
from src.data_utils import load_gold_doc_ids
from src.generation import (
    load_generator, build_rag_prompt,
    generate_answer, answer_loss_and_perplexity, estimate_model_size_mb
)
from src.metrics import (
    evaluate_qa_predictions, recall_at_k, support_recall_at_k, numeric_summary
)
from src.retrieval import DenseRetriever, format_context, retrieved_doc_ids
from src.analysis_utils import Timer, save_json

pd.set_option("display.max_colwidth", 160)

In [4]:
questions_df = pd.read_parquet(PROCESSED_DIR / "questions.parquet")
corpus_df = pd.read_parquet(PROCESSED_DIR / "corpus.parquet")

EVAL_N = min(2000, len(questions_df))
TOP_K = 5

eval_df = questions_df.head(EVAL_N).copy()
print(eval_df.shape)

(2000, 7)


In [5]:
dense_retriever = DenseRetriever(
    index_path=INDEX_DIR / "dense_faiss.index",
    doc_ids_path=INDEX_DIR / "dense_doc_ids.json",
    corpus_df=corpus_df,
    model_name=EMBEDDING_MODEL,
)

tokenizer, model, device = load_generator(GENERATOR_MODEL)
print("Device:", device)
print("Approx model size MB:", estimate_model_size_mb(model))

Device: mps
Approx model size MB: 1884.58544921875


In [6]:
rows = []

for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Vanilla RAG"):
    gold_doc_ids = load_gold_doc_ids(row["gold_doc_ids"])

    with Timer() as rt:
        chunks = dense_retriever.retrieve(row["question"], top_k=TOP_K)
    retrieval_latency = rt.elapsed

    ret_doc_ids = retrieved_doc_ids(chunks)
    context = format_context(chunks)
    prompt = build_rag_prompt(row["question"], context)

    pred, gen_latency = generate_answer(
        tokenizer=tokenizer,
        model=model,
        prompt=prompt,
        max_new_tokens=48,
        temperature=0.0,
    )

    ppl_info = answer_loss_and_perplexity(
        tokenizer=tokenizer,
        model=model,
        prompt=prompt,
        answer=row["answer"],
    )

    rows.append({
        "system": "vanilla_rag_dense",
        "sample_id": row["sample_id"],
        "question": row["question"],
        "answer": row["answer"],
        "prediction": pred,
        "answer_loss": ppl_info["answer_loss"],
        "answer_ppl": ppl_info["answer_ppl"],
        "answer_n_tokens": ppl_info["answer_n_tokens"],
        "retrieved_doc_ids": json.dumps(ret_doc_ids, ensure_ascii=False),
        "gold_doc_ids": json.dumps(gold_doc_ids, ensure_ascii=False),
        "retrieval_hit_at_5": recall_at_k(ret_doc_ids, gold_doc_ids, 5),
        "support_recall_at_5": support_recall_at_k(ret_doc_ids, gold_doc_ids, 5),
        "retrieval_latency_sec": retrieval_latency,
        "generation_latency_sec": gen_latency,
        "total_latency_sec": retrieval_latency + gen_latency,
    })

pred_df = pd.DataFrame(rows)
pred_df.to_csv(PREDICTIONS_DIR / "vanilla_rag_predictions.csv", index=False)

display(pred_df.head())

Vanilla RAG:   0%|          | 0/2000 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


,system,sample_id,question,answer,prediction,answer_loss,answer_ppl,answer_n_tokens,retrieved_doc_ids,gold_doc_ids,retrieval_hit_at_5,support_recall_at_5,retrieval_latency_sec,generation_latency_sec,total_latency_sec
0,vanilla_rag_dense,5a8b57f25542995d1e6f1371,Were Scott Derrickson and Ed Wood of the same nationality?,yes,No.,5.074569,159.903219,1,"[""5a8b6d885542997f31a41d25::9::Ed Wood (film)"", ""5a8b57f25542995d1e6f1371::0::Ed Wood (film)"", ""5a84d2ea5542992a431d1aa8::0::Ed Wood"", ""5a8b57f25542995d1e6f...","[""5a8b57f25542995d1e6f1371::1::Scott Derrickson"", ""5a8b57f25542995d1e6f1371::4::Ed Wood""]",1.0,0.5,0.179618,2.340463,2.520081
1,vanilla_rag_dense,5a8c7595554299585d9e36b6,What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?,Chief of Protocol,"To determine the government position held by the woman who portrayed Corliss Archer in the film Kiss and Tell, let's analyze the information step-by-step:",8.324216,4122.503440,3,"[""5a8c7595554299585d9e36b6::6::Kiss and Tell (1945 film)"", ""5ae1b1445542997283cd223d::7::A Kiss for Corliss"", ""5a8c7595554299585d9e36b6::5::A Kiss for Corli...","[""5a8c7595554299585d9e36b6::1::Shirley Temple"", ""5a8c7595554299585d9e36b6::6::Kiss and Tell (1945 film)""]",1.0,0.5,0.040785,2.081338,2.122123
2,vanilla_rag_dense,5a85ea095542994775f606a8,"What science fantasy young adult series, told in first person, has a set of companion books narrating the stories of enslaved worlds and alien species?",Animorphs,The Chronicles of Tornor,5.811562,334.140507,3,"[""5abeebe25542993fe9a41d9c::6::First North Americans"", ""5a73024f5542991f9a20c5fc::5::Multiverse: Exploring Poul Anderson's Worlds"", ""5a85ea095542994775f606a...","[""5a85ea095542994775f606a8::2::The Hork-Bajir Chronicles"", ""5a85ea095542994775f606a8::8::Animorphs""]",0.0,0.0,0.246592,2.627659,2.874251
3,vanilla_rag_dense,5adbf0a255429947ff17385a,Are the Laleli Mosque and Esma Sultan Mansion located in the same neighborhood?,no,No.,3.145353,23.227881,1,"[""5adbf0a255429947ff17385a::6::Esma Sultan Mansion"", ""5adbf0a255429947ff17385a::5::Laleli Mosque"", ""5a88a35c554299206df2b316::2::Abidin Mosque"", ""5adbf0a255...","[""5adbf0a255429947ff17385a::5::Laleli Mosque"", ""5adbf0a255429947ff17385a::6::Esma Sultan Mansion""]",1.0,1.0,0.052925,2.245032,2.297957
4,vanilla_rag_dense,5a8e3ea95542995a26add48d,"The director of the romantic comedy ""Big Stone Gap"" is based in what New York city?","Greenwich Village, New York City",New York City,2.243537,9.426616,6,"[""5a8e3ea95542995a26add48d::9::Big Stone Gap (film)"", ""5ade15f35542997545bbbe47::0::Manhattan Romance"", ""5ae0bae555429906c02dab08::9::Stone Stanley Entertai...","[""5a8e3ea95542995a26add48d::3::Adriana Trigiani"", ""5a8e3ea95542995a26add48d::9::Big Stone Gap (film)""]",1.0,0.5,0.044907,2.042526,2.087433


In [ ]:
metrics = {
    "system": "vanilla_rag_dense_top5",
    "top_k": TOP_K,
}

metrics.update(evaluate_qa_predictions(rows))

metrics.update(numeric_summary(pred_df["answer_loss"], "answer_loss"))
metrics.update(numeric_summary(pred_df["answer_ppl"], "answer_ppl"))
metrics.update(numeric_summary(pred_df["retrieval_latency_sec"], "retrieval_latency_sec"))
metrics.update(numeric_summary(pred_df["generation_latency_sec"], "generation_latency_sec"))
metrics.update(numeric_summary(pred_df["total_latency_sec"], "total_latency_sec"))

metrics["model_name"] = GENERATOR_MODEL
metrics["model_size_mb_estimate"] = estimate_model_size_mb(model)
save_json(METRICS_DIR / "vanilla_rag_metrics.json", metrics)

metrics

{'system': 'vanilla_rag_dense_top5',
 'top_k': 5,
 'exact_match': 0.106,
 'token_f1': 0.1848929393427386,
 'contains_answer': 0.255,
 'n': 2000,
 'answer_loss_mean': 3.2459398184149757,
 'answer_loss_median': 2.517337203025818,
 'answer_loss_p90': 7.283771944046026,
 'answer_loss_p95': 9.05986008644104,
 'answer_loss_max': 16.93240737915039,
 'answer_ppl_mean': 52692.79185757768,
 'answer_ppl_median': 12.395587449858773,
 'answer_ppl_p90': 1456.5702181014103,
 'answer_ppl_p95': 8602.974369926964,
 'answer_ppl_max': 22576212.804087013,
 'retrieval_latency_sec_mean': 0.03584547648651109,
 'retrieval_latency_sec_median': 0.026162354500229412,
 'retrieval_latency_sec_p90': 0.0525963755997509,
 'retrieval_latency_sec_p95': 0.09111085029994676,
 'retrieval_latency_sec_max': 1.7221605000004274,
 'generation_latency_sec_mean': 2.7327798599039914,
 'generation_latency_sec_median': 2.5922704160000194,
 'generation_latency_sec_p90': 3.30944951670017,
 'generation_latency_sec_p95': 3.6300407381500

### Parametric-only vs Vanilla RAG

The vanilla dense RAG system substantially improves performance over the parametric-only baseline on the expanded 2,000-example evaluation. Exact Match increases from `0.0255` to `0.1060`, token-level F1 increases from `0.0869` to `0.1849`, and contains-answer increases from `0.1455` to `0.2550`. This shows that retrieved evidence helps the model generate answers that more often include the correct entity or phrase.

The language-modeling metrics show the same trend. Mean answer loss decreases from `3.945` to `3.246`, and median answer perplexity decreases from `29.65` to `12.40`. Because mean perplexity is dominated by outliers, the most reliable comparison is the reduction in mean answer loss and the decrease in median/p90 perplexity.

The improvement comes with a latency cost, but the retrieval step itself is cheap. Average total latency increases from `2.11` seconds to `2.77` seconds per query. Dense retrieval takes only `0.036` seconds on average, while generation latency increases from `2.11` seconds to `2.73` seconds because the generator receives a longer prompt with retrieved context. Therefore, in this setup, the main runtime cost of vanilla RAG is longer-context generation rather than FAISS search.